In [6]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils import load_config, unscale_masses
from src.validation import get_tpcf

In [7]:
# In and Out Directories
train_run = "20260508_154857"
infer_run = "20260510_140248"
config_dir = f"/gpfs/home4/bartb/T256/T256-SUBBOX/results/run_{train_run}/qualitative_2pcf/{infer_run}/final_config.yaml"
config = load_config(config_dir)
output_dir = os.path.dirname(config_dir)

In [8]:
# Load generated masses
gen_masses = torch.load(f"/gpfs/home4/bartb/T256/T256-SUBBOX/results/run_{train_run}/qualitative_2pcf/{infer_run}/gen_masses.pth", weights_only=False)
print(f"Number of cosmologies: {len(gen_masses)}")

Number of cosmologies: 5


In [11]:
# Descale masses
log10_masses = [unscale_masses(m) for m in gen_masses]

In [13]:
n_eval = 5
mass_bins = np.logspace(13.5, 16.5, 50)  # mass bins in h^-1 M_sun
box_volume = 370.**3  # (h^-1 Mpc)^3

hmf_true_avg = []
hmf_true_std = []

for i in range(n_eval):
    indices = subbox_indices_per_cosm[i]
    hmfs = []
    for idx in indices:
        n_h = halo_counts[idx]
        # Column 6 in the full data is mass — check your data format
        raw_masses = np.load("/gpfs/home4/bartb/T256/Data/subboxes/test_subbox_halos.npy",
                            mmap_mode='r')[idx, :n_h, 6]
        # If masses are already scaled [0,1], unscale:
        physical_masses = unscale_mass(raw_masses)
        # Cumulative: n(> M)
        cumulative = np.array([np.sum(physical_masses > m) for m in mass_bins]) / box_volume
        hmfs.append(cumulative)
    
    hmfs = np.array(hmfs)
    hmf_true_avg.append(np.mean(hmfs, axis=0))
    hmf_true_std.append(np.std(hmfs, axis=0))

hmf_true_avg = np.array(hmf_true_avg)
hmf_true_std = np.array(hmf_true_std)

NameError: name 'subbox_indices_per_cosm' is not defined